In [ ]:
!nvidia-smi

In [ ]:
import os
HOME = os.getcwd()
print(HOME)

In [ ]:
# Cách cài đặt bằng pip (khuyến nghị)

!pip install ultralytics==8.0.20

from IPython import display
display.clear_output()

import ultralytics
ultralytics.checks()


In [ ]:
# Cách clone bằng Git (phù hợp khi phát triển)

# %cd {HOME}
# !git clone github.com/ultralytics/ultralytics
# %cd {HOME}/ultralytics
# !pip install -qe ultralytics

# from IPython import display
# display.clear_output()

# import ultralytics
# ultralytics.checks()


In [ ]:
from ultralytics import YOLO

from IPython.display import display, Image

## Cơ bản về CLI


Nếu bạn muốn huấn luyện, đánh giá hoặc chạy suy luận với mô hình mà không cần chỉnh sửa mã nguồn, giao diện dòng lệnh YOLO là cách nhanh nhất để bắt đầu. Xem thêm về CLI tại [tài liệu Ultralytics YOLO](https://v8docs.ultralytics.com/cli/).

```
yolo task=detect    mode=train    model=yolov8n.yaml      args...
          classify       predict        yolov8n-cls.yaml  args...
          segment        val            yolov8n-seg.yaml  args...
                         export         yolov8n.pt        format=onnx  args...
```


## Suy luận với mô hình COCO huấn luyện sẵn


### 💻 Giao diện dòng lệnh (CLI)


`yolo mode=predict` chạy suy luận YOLOv8 trên nhiều loại dữ liệu đầu vào, tự động tải mô hình từ bản phát hành YOLOv8 mới nhất và lưu kết quả vào `runs/predict`.


In [ ]:
%cd {HOME}
!yolo task=detect mode=predict model=yolov8n.pt conf=0.25 source='https://media.roboflow.com/notebooks/examples/dog.jpeg' save=True

In [ ]:
%cd {HOME}
Image(filename='runs/detect/predict/dog.jpeg', height=600)

### 🐍 Python SDK

Đây là cách đơn giản nhất để sử dụng trực tiếp YOLOv8 trong môi trường Python.


In [ ]:
model = YOLO(f'{HOME}/yolov8n.pt')
results = model.predict(source='https://media.roboflow.com/notebooks/examples/dog.jpeg', conf=0.25)

In [ ]:
results[0].boxes.xyxy

In [ ]:
results[0].boxes.conf

In [ ]:
results[0].boxes.cls

In [ ]:
!rmdir {HOME}/datasets
!mkdir {HOME}/datasets
%cd {HOME}/datasets

!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="fkzZLLRJRWlKMXk3M0o1")
project = rf.workspace("projects-uv0zk").project("safety-helmet-detection-ttig7")
dataset = project.version(1).download("yolov8")

## Huấn luyện mô hình tùy chỉnh


In [ ]:
%cd {HOME}

!yolo task=detect mode=train model=yolov8s.pt data={dataset.location}/data.yaml epochs=100 imgsz=800 plots=True

In [ ]:
!ls {HOME}/runs/detect/train/

In [ ]:
%cd {HOME}
Image(filename=f'{HOME}/runs/detect/train/confusion_matrix.png', width=600)

In [ ]:
%cd {HOME}
Image(filename=f'{HOME}/runs/detect/train/results.png', width=600)

In [ ]:
%cd {HOME}
Image(filename=f'{HOME}/runs/detect/train/val_batch0_pred.jpg', width=600)

## Đánh giá mô hình tùy chỉnh


In [ ]:
%cd {HOME}

!yolo task=detect mode=val model={HOME}/runs/detect/train/weights/best.pt data={dataset.location}/data.yaml

## Suy luận với mô hình tùy chỉnh


In [ ]:
%cd {HOME}
!yolo task=detect mode=predict model={HOME}/runs/detect/train/weights/best.pt conf=0.25 source={dataset.location}/test/images save=True

**Lưu ý:** Hãy xem qua một vài kết quả mẫu.


In [ ]:
import glob
from IPython.display import Image, display

for image_path in glob.glob(f'{HOME}/runs/detect/predict2/*.jpg')[:3]:
      display(Image(filename=image_path, width=600))
      print("\n")

## Triển khai mô hình lên Roboflow

Sau khi huấn luyện xong mô hình YOLOv8, bạn sẽ có bộ trọng số đã sẵn sàng để sử dụng. Các trọng số này nằm trong thư mục `/runs/detect/train/weights/best.pt` của dự án. Bạn có thể tải trọng số mô hình lên Roboflow Deploy để sử dụng trên hạ tầng có khả năng mở rộng cao của Roboflow.

Hàm `.deploy()` trong [gói `roboflow` cho Python](https://docs.roboflow.com/python) hiện hỗ trợ tải trọng số YOLOv8 lên nền tảng này.

Để tải trọng số mô hình lên, hãy thêm đoạn mã sau vào phần “Suy luận với mô hình tùy chỉnh” trong notebook:


In [ ]:
project.version(dataset.version).deploy(model_type="yolov8", model_path=f"{HOME}/runs/detect/train/")

In [ ]:
# Trong lúc quá trình triển khai đang chạy, bạn có thể xem tài liệu deploy để đưa mô hình tới nhiều môi trường khác nhau: https://docs.roboflow.com/inference


In [ ]:
# Chạy suy luận với mô hình của bạn trên API đám mây có khả năng tự mở rộng

# tải mô hình
model = project.version(dataset.version).model

# chọn ngẫu nhiên một ảnh trong tập kiểm tra
import os, random
test_set_loc = dataset.location + "/test/images/"
random_test_image = random.choice(os.listdir(test_set_loc))
print("Đang chạy suy luận trên " + random_test_image)

pred = model.predict(test_set_loc + random_test_image, confidence=40, overlap=30).json()
pred


In [ ]:
from google.colab import files
files.download('/content/runs/segment/train3/weights/best.pt')